In [1]:
!uv add qdrant-client[fastembed]

Resolved 266 packages in 15ms
Checked 258 packages in 71ms


In [2]:
from qdrant_client import QdrantClient, models

qd_client = QdrantClient(":memory:")

In [3]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
dimensionality = 384
collection_name = "evidently-docs"


# if you want to re-insert the data, delete the collection first
# qd_client.delete_collection(collection_name=collection_name)


qd_client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=dimensionality,
        distance=models.Distance.COSINE
    )
)


True

In [4]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()
parsed_docs = [doc.parse() for doc in files]
doc_chunks = chunk_documents(parsed_docs, size=3000, step=1500)

In [7]:
points = []

for i, doc in enumerate(doc_chunks):
    text = doc.get('title', '') + ' ' + doc.get('description', '') + ' ' + doc['content']
    text = text.strip()
    vector = models.Document(text=text, model=model_name)
    point = models.PointStruct(
        id=i,
        vector=vector,
        payload=doc
    )
    points.append(point)


qd_client.upsert(
    collection_name=collection_name,
    points=points
)



UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [10]:
def vector_search(query, num_results=5):    
    vector = models.Document(text=query, model=model_name)

    query_points = qd_client.query_points(
        collection_name=collection_name,
        query=vector,
        limit=num_results,
        with_payload=True
    )

    results = []
    for point in query_points.points:
        results.append(point.payload)

    return results

results = vector_search("how do I create a dashboard?")
for r in results:
    print(r.get('title', 'No title'))

Add dashboard panels (API)
Add dashboard panels (UI)
Add dashboard panels (UI)
No title
Overview


In [11]:
def search(query):
    return vector_search(query, num_results=5)

In [12]:
search("how do I create a dashboard?")

[{'start': 0,
  'content': 'You can add Panels in the user interface or using Python API. This pages describes the Python API. Check how to [add panels in the UI](dashboard_add_panels_ui).\n\n## Dashboard Management\n\n<Check>\n  Dashboards as code are available in Evidently OSS, Cloud, Enterprise.\n</Check>\n\n<Tip>\n  You must first connect to [Evidently Cloud](/docs/setup/cloud) and [create a Project](/docs/platform/projects_manage).\n</Tip>\n\n**Adding Tabs**. To add a new Tab:\n\n```python\nproject.dashboard.add_tab("Another Tab")\n```\n\nYou can also create a new Tab while adding a Panel as shown below. If the destination Tab doesn\'t exist, it will be created. If it does, the Panel will be added below existing ones in that Tab.\n\n**Deleting Tabs**. To delete a Tab:\n\n```python\nproject.dashboard.delete_tab("Another Tab")\n```\n\n**Deleting Panels**. To delete a specific Panel:\n\n```python\nproject.dashboard.delete_panel("Dashboard title", "My new tab")\n```\n\n(First list the